In [0]:
df_Products = spark.read.csv("/Volumes/workspace/bronze/retailsalesanalytics/products.csv",header=True,inferSchema=True)
df_Products.display()


In [0]:
df_Products.write.format("Delta").mode("overwrite").saveAsTable("bronze_db.Products_bronze")

In [0]:
df_Sales = spark.read.csv("/Volumes/workspace/bronze/retailsalesanalytics/sales.csv",header=True)
df_Sales.display()

In [0]:
df_Sales.write.format("Delta").mode("overwrite").saveAsTable("bronze_db.Sales_bronze")

In [0]:
df_Stores = spark.read.csv("/Volumes/workspace/bronze/retailsalesanalytics/stores.csv",header=True,inferSchema=True)
df_Stores.display()

In [0]:
df_Stores.write.format("Delta").mode("overwrite").saveAsTable("bronze_db.Stores_bronze")


In [0]:
df_Products.printSchema()
df_Sales.printSchema()
df_Stores.printSchema()

In [0]:
# remove invalid rows: missing product_id, store_id, or quantity
remove_rows = df_Stores.na.drop(subset=["product_id"])
products_silver = remove_rows 

In [0]:
#Remove duplicate sale_id record
df_remove = df_Sales.dropDuplicates(["sale_id"])


In [0]:
#Convert data types: quantity to integer, price to integer, sale_timestamp to timestamp
from pyspark.sql.functions import col
Convert_Data_type = df_remove.withColumn("Quantity_int", col("quantity").cast("int"))\
    .withColumn("price_int", (col("price")).cast("int"))\
    .withColumn("Sale_timestamp_to_timestamp", (col("sale_timestamp")).cast("timestamp"))



In [0]:
#Create calculated columns: total_amount = quantity × price, sale_date from sale_timestamp
from pyspark.sql import functions as F
total_amount = Convert_Data_type.withColumn("total_amount", F.col("Quantity_int") * F.col("price_int"))\
    .withColumn("sale_date", F.to_date(F.col("sale_timestamp")))
total_amount.display()

In [0]:
# Validate cleaned data: check for negative quantity, zero price, future timestamps
from pyspark.sql import functions as F

validated_data_df = total_amount.filter(
    (F.col("Quantity_int") >= 0) &
    (F.col("price_int") > 0) 
)
display(validated_data_df)
